In the following discussion, we will equate a Pokemon with its type, stats, and four known moves.  There's more to it, of course, but this seems like a good place to start.

# The Problem

A pokemon's type, stats, and four known moves gives 12 dimensions of information about that pokemon.  While Player 1 win probability should be a nondecreasing function of the numeric stats, the amount of benefit provided from increasing a stat by one point depends on the matchup.  If you replace Weavile by Chien-Pao, your win probability should go up!  But how much it goes up should depend on the matchup.  If your opponent has no pokemon between Weavile's base 125 speed and Chien-Pao's base 135 speed, then your win probability shouldn't go up by very much.  Or, if your oppenent has lots of pokemon with high physical defense, then Chien-Pao's power increase (coming from its ability, not its stats) may not make much of a difference either.  But if your opponent has Eternatus (base speed 130), then replacing Weavile by Chien-Pao should make a big difference!  So a statistic's contribution to win probability is matchup dependent, and this seems hard for a model to learn.

Another issue is that for nearly every pokemon (maybe we should quantify this claim), their secondary attacking stat is irrelevant.  It does not matter what your Chien-Pao's special attack stat is; it will not affect the battle at all.  So when we're training a model, the model should not be told Chien-Pao's special attack stat.  One possibility is to replace every pokemon's attack and special attack stat with offensive_stat = max(atk, spa) and record a boolean offensive_stat_is_phys.  This is okay, but it seems like the model might have a hard time learning that if offensive_stat_is_phys == True, then offensive_stat should be compared with its opponents def stats, not spd stats.

Yet another issue is that the model will have to somehow divine the type chart from these battle outcomes.  This isn't a situation where having more Psychic types is better than having fewer; once again, it's matchup-dependent.

# A Solution

Rather than giving the model a bunch of basic information (types, stats, known moves) and hoping that the model learns the stuff we already know (Fire > Grass > Water > Fire), let's tell the model the stuff that we already know.  But how do you feed in the type chart and whatnot into the model?  Seems hard.

Instead, let's bake all of our knowledge into a set of offensive and defensive stats (we'll call these "advantage" stats) that are derived for each specific battle.  Then, we'll replace the basic stats with these "advantage" stats.

We will start off by neglecting the movepool.  We could try to map each move to its category, type, and base power, but that could be a lot of work.  It's something to explore in the future.

## Damage Approximator

The advantage stat starts with an approximation of damage.  Given a pokemon mon1 on team 1 and a pokemon mon2 on team 2, we can estimate the average damage that mon1 would do to mon2 by selecting its best STAB move (or a not-very-effective coverage move in the event that is better--this could be updated if we get data suggesting that in the event that a mon's best move is a coverage move, the coverage move is often neutral or better).  Note that we assume a base power of 80 for all moves.

We approximate that when mon1 attacks mon2, mon2 will lose the following fraction of its HP:

$$\text{damage}(\text{mon1},\text{mon2}) := \left(\frac{\left(\frac{2 \cdot \text{mon1}_{\text{level}}}{5} + 2\right) \cdot 80 \cdot \frac{\text{mon1}_{\text{off-stat}}}{\text{mon2}_{\text{def-stat}}}}{50} + 2\right) \cdot \text{type-multiplier}(\text{mon1},\text{mon2}) \cdot 92.5/100 / \text{mon2}_{\text{hp}}$$

where 

$$\text{mon1}_{\text{off-stat}} = \max(\text{mon1}_{\text{atk}}, \text{mon1}_{\text{spa}}),$$

$$ \text{mon2}_{\text{def-stat}} = \begin{cases} \text{mon2}_{\text{def}} & \text{if } \text{mon1}_{\text{off-stat}} = \text{mon1}_{\text{atk}} \\ \text{mon2}_{\text{spd}} & \text{else} \end{cases}, $$

$$ \text{type-multiplier}(\text{mon1},\text{mon2}) = \max\left(\frac{1}{2}, \max_{\text{type1} \in \text{mon1}_{\text{types}}} 1.5 \cdot \prod_{\text{type2} \in \text{mon2}_{\text{types}}} \text{eff}(\text{type1},\text{type2})\right) $$

with $\text{eff}(\text{type1},\text{type2})$ being determined by the type chart, so that, e.g. $\text{eff}(\text{water},\text{fire}) = 2$ while $\text{eff}(\text{fire},\text{water}) = 1/2$.

Some notes:
- This fraction is allowed to exceed 1 as the amount by which it exceeds 1 may actually matter (think: reflect/light screen/aurora veil or resistance berries).
- This uses a simplified version of the damage formula found here: https://bulbapedia.bulbagarden.net/wiki/Damage#Generation_V_onward
- The number 80 corresponds to the base power of the move theoretically being selected.
- The use of 92.5/100 corresponds to the average value of the 'random' factor.
- The use of type-multiplier is meant to approximate the product of STAB and Type.
- type-multiplier assumes that mon1 is only using STABs.  This could be updated to account for coverage moves in a future iteration on this stat.
- The max(1/2, stuff) in type-multiplier is to prevent type-multiplier from ever being 0.  It is very rare (though it does happen) that mon1 will be unable to damage mon2.  The factor of 1/2 is used because that is a multiplier for a not-very-effective coverage move.  This would be resolved by replacing the max over mon1 types by a max over mon1 move types.
- Damage or speed boosting items will not be accounted for.  This could be resolved in an ad-hoc way by checking for common boosting items (choice items, life orb).  It could also be resolved in a systemic way be using the smogon damage calculator to replace the offensive advantage stat.
- Damage/stat modifying abilities like Levitate, Thick Fat, or Sword of Ruin will not be accounted for.  This could 'only' be resolved by using the smogon damage calculator to replace this offensive advantage stat.
- Type chart modifying moves like Freeze-Dry are not accounted for.


## Advantage

The only (relevant) thing that doesn't go into the damage approximator is speed.  Speed is difficult to incorporate into advantage.  There are a few reasons for this:
  1. The only important feature of speed differential (meaning $\text{mon1}_{\text{spe}} - \text{mon2}_{\text{spe}}$) is its sign.  Magnitude is meaningless here.  So multiplying the damage approximation by speed differential would be a bad idea.
  2. The impact of speed differential can be large or small.  If you consider a hypothetical Weavile versus Iron Boulder matchup, each has a super-effective STAB on the other (meaning it has a type-multiplier equal to 3)!  In that situation, Weavile has the advantage because it goes first.  However, if you consider a Weavile versus Swampert matchup (where each has a type-multiplier of 1.5), the Swampert has the advantage in spite of its speed disadvantage due to its overall bulk.  My initial thought is that speed matters a lot when both pokemon are doing about the same amount of damage to one another, but doesn't matter very much when the pokemon are doing very different amounts of damage.  So 'having a speed advantage' should not correspond to a constant factor.

Also worth noting is that advantage depends not just on how much damage you're doing to your opponent, but how much damage your opponent is doing to you!

Maybe try computing 'turns to KO' for each mon and look at differential.  So we get something like

$$ \text{time-to-ko-diff}(\text{mon1},\text{mon2}) = \lfloor \frac{1}{\min(1,\text{damage-approx}(\text{mon2},\text{mon1}))}\rfloor - \lfloor\frac{1}{\min(1,\text{damage-approx}(\text{mon1},\text{mon2}))}\rfloor $$

Here, bigger is better for mon1.  Then perhaps, add 1/2 to the faster mon's score and subtract 1/2 from the slower mon's score so that the faster mon has a 1 turn advantage?

Properties that I want:
- There should be a nice relationship between adv(mon1,mon2) and adv(mon2,mon1) (a + b = 1 with 0 < a,b < 1?  ab = 1?)
- If time-to-ko-diff is close to 0 and the times to ko are close to 1, the faster mon should have a big advantage (this is the situation where the faster mon just OHKOs the slower mon with no cost)
- If the time-to-ko-diff is close to 0 and the times to ko are large, the faster mon should have a small advantage (this is the situation where the faster mon KOs the slower mon but at an HP cost)
- If the time-to-ko-diff is large, the mon with the smaller time to ko should have a big advantage (this is the situation where one mon just dominates the other)
- If the time-to-ko-diff is 

So maybe the advantage stat should represent something like: expected damage dealt to opponent in a 1v1 matchup?  If we let $n$ denote the round number in which the KO occurs, then the faster mon gets to go $n$ times and the slower mon gets to go $n$ or $n-1$ times depending on who wins.

Let's set

$$\text{time-to-ko}(\text{mon1},\text{mon2}) = \lfloor \frac{1}{\min(1,\text{damage-approx}(\text{mon1},\text{mon2}))} \rfloor$$

Then

$$\text{turn-of-ko}(\text{mon1},\text{mon2}) = \min\left(\text{time-to-ko}(\text{mon1},\text{mon2}), \text{time-to-ko}(\text{mon2},\text{mon1})\right)$$

So

$$\text{1v1-damage}(\text{mon1},\text{mon2}) =
\begin{cases}
    \text{turn-of-ko}(\text{mon1},\text{mon2}) \cdot \text{damage-approx}(\text{mon1},\text{mon2})  &\text{if } \text{mon1}_{\text{spe}} > \text{mon2}_{\text{spe}}\\
    (\text{turn-of-ko}(\text{mon1},\text{mon2}) - 1) \cdot \text{damage-approx}(\text{mon1},\text{mon2})  &\text{if } \text{mon1}_{\text{spe}} < \text{mon2}_{\text{spe}}
\end{cases}
$$
How to handle speed ties?

Then we can do something like set $\text{adv}(\text{mon1},\text{mon2}) = \text{1v1-damage}(\text{mon1},\text{mon2})$ or we can do something fancy and make it symmetric like $\text{adv}(\text{mon1},\text{mon2}) = \text{1v1-damage}(\text{mon1},\text{mon2}) - \text{1v1-damage}(\text{mon2},\text{mon1})$.

Regardless, this should be enough to get started.


## Potential problems with advantage stats

Some pokemon are not good because of their stats.  Take Sableye for example.  It has atrocious stats, but can win a match on the strength of its ability, Prankster.  These advantage stats won't account for that.  (On the other hand, neither will training on 12-dimensional info above.)

Other pokemon don't rely on their offensive stats for damage (think Toxapex).